In [ ]:
import torch
from torch.utils.data import DataLoader, random_split
from tqdm.auto import tqdm
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

In [14]:
CE = torch.nn.CrossEntropyLoss()

def run_epoch(model, loader, optim, device, train=True):
    model.train() if train else model.eval()
    all_preds = []
    all_labels = []
    total_loss = 0.0
    batches = 0

    loop = tqdm(loader, desc='Train' if train else 'Val', leave=False)
    for input_ids, labels in loop:
        input_ids, labels = input_ids.to(device), labels.to(device)

        if train:
            optim.zero_grad()

        with torch.set_grad_enabled(train):
            outputs = model(input_ids)
            loss = CE(outputs, labels)
            if train:
                loss.backward()
                optim.step()

        total_loss += loss.item()
        batches += 1

        preds = outputs.argmax(dim=1).cpu().tolist()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().tolist())

    acc  = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, zero_division=0)
    rec  = recall_score(all_labels, all_preds, zero_division=0)
    f1   = f1_score(all_labels, all_preds, zero_division=0)
    avg_loss = total_loss / batches

    if train:
        print(f"[Train] Loss: {avg_loss:.4f}  Acc: {acc:.4f}  Prec: {prec:.4f}  Rec: {rec:.4f}  F1: {f1:.4f}")
    else:
        print(f"[Val]   Loss: {avg_loss:.4f}  Acc: {acc:.4f}  Prec: {prec:.4f}  Rec: {rec:.4f}  F1: {f1:.4f}")

    return {
        'loss':   avg_loss,
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1': f1
    }

In [8]:
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer
import re

def remove_lines(text: str) -> str:
    cleaned_lines = []
    for line in text.splitlines():
        stripped = line.strip()
        if not stripped:
            continue
        if stripped == "Низкие комиссии. Ежедневная подборка инвестиционных идей":
            continue
        if re.fullmatch(r"[\-,]+", stripped):
            continue
        cleaned_lines.append(line)
    return "\n".join(cleaned_lines)

In [12]:
class FusionDataset(Dataset):
    def __init__(self, path:str):
        df = pd.read_csv(path)
        self.tok = AutoTokenizer.from_pretrained("ai-forever/ruBERT-base")
        self.samples = []
        df['title'].fillna('', inplace=True)
        df['text'].fillna('', inplace=True)

        for _, row in df.iterrows():
            full_text = remove_lines(f"{row['title']} {row['text']}")
            enc = self.tok(full_text,
                           truncation=True,
                           padding='max_length',
                           max_length=128,
                           return_tensors='pt')
            input_ids = enc.input_ids.squeeze(0)

            label = torch.tensor(int(row['label']), dtype=torch.long)

            self.samples.append((input_ids, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

In [ ]:
from transformers import AutoModel
import torch.nn as nn

class FusionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.text_model = AutoModel.from_pretrained("ai-forever/ruBERT-base")
        self.classifier = nn.Linear(self.text_model.config.hidden_size, 2)

    def forward(self, input_ids):
        out = self.text_model(input_ids=input_ids)
        pooled = out.last_hidden_state[:, 0]
        return self.classifier(pooled)

In [16]:
dataset = FusionDataset('fusion_dataset.csv')

dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_ds, val_ds = random_split(dataset, [int(0.8 * len(dataset)), len(dataset) - int(0.8 * len(dataset))])
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=8)


model = FusionModel() 
model.to(dev)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5, weight_decay=1e-2)

for epoch in range(1, 4):
    run_epoch(model, train_loader, optimizer, dev, train=True)
    run_epoch(model, val_loader, optimizer, dev, train=False)

c:\Users\andrey\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Train:   0%|          | 0/1261 [00:00<?, ?it/s]

[Train] Loss: 0.7043  Acc: 0.5040  Prec: 0.5155  Rec: 0.5538  F1: 0.5340


Val:   0%|          | 0/316 [00:00<?, ?it/s]

[Val]   Loss: 0.7007  Acc: 0.4976  Prec: 0.4976  Rec: 1.0000  F1: 0.6645


...

[Val]   Loss: 0.6863  Acc: 0.5381  Prec: 0.5320  Rec: 0.5960  F1: 0.5622


Train:   0%|          | 0/1261 [00:00<?, ?it/s]

[Train] Loss: 0.6676  Acc: 0.5915  Prec: 0.5965  Rec: 0.6296  F1: 0.6126


Val:   0%|          | 0/316 [00:00<?, ?it/s]

[Val]   Loss: 0.7098  Acc: 0.5333  Prec: 0.5190  Rec: 0.8510  F1: 0.6447
